# Corner Annotation & Correction

Workflow for auditing and correcting `board_corner` annotations in the v1 dataset.

**Sections:**
1. Audit — identify images with suspicious corner annotations
2. Prepare workspace — export flagged images for the annotator tool
3. Launch annotator — open the HTML/JS annotation UI in the browser
4. Review corrections — inspect what changed and apply to dataset

In [1]:
%load_ext autoreload
%autoreload 2

import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from IPython.display import display

from moku.dataset import (
    CATEGORIES,
    ID_TO_CATEGORY,
    apply_corner_corrections,
    audit_corners,
)

# ── Configuration ─────────────────────────────────────────────────────────
HF_DATASET    = "kaya-go/moku-v1"
ANNOTATE_DIR  = Path("data/annotate")      # local workspace for the annotator
CORRECTED_FILE = ANNOTATE_DIR / "corrected.json"
SERVER_PORT    = 7860

ANNOTATE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Workspace: {ANNOTATE_DIR.resolve()}")

Workspace: /Users/hadim/Code/libs/moku/notebooks/data/annotate


## 1. Audit Corner Annotations

Checks each image for:
- Wrong number of corners (expected 4)
- Corners not distributed across 4 distinct quadrants
- Unusual bbox sizes (too small / too large relative to image area)

In [2]:
dataset = load_dataset(HF_DATASET)
print(f"Dataset: {HF_DATASET}")
for split, ds in dataset.items():
    print(f"  {split}: {len(ds)} images")

Dataset: kaya-go/moku-v1
  train: 382 images
  validation: 53 images
  test: 50 images


In [3]:
issues_df = audit_corners(dataset, expected_count=4)

print(f"\nFlagged images: {len(issues_df)} / {sum(len(ds) for ds in dataset.values())} total")
if len(issues_df) > 0:
    issue_counts = issues_df["issues"].str.split(", ").explode().value_counts()
    print("\nIssue breakdown:")
    display(issue_counts.rename("count").to_frame())
    print("\nFlagged images:")
    display(issues_df)


Flagged images: 133 / 485 total

Issue breakdown:


,count
issues,
not_in_4_quadrants,115
'TR']),110
'TL',87
duplicate_quadrant (['BL',69
duplicate_quadrant (['BR',35
wrong_count (3 vs 4),18
duplicate_quadrant (['TL',11
'BR',10
'TL']),3



Flagged images:


,split,image_id,source_dataset,n_corners,issues
0,train,7,go_game_v10,4,"duplicate_quadrant (['BL', 'TL', 'TR']), not_i..."
1,train,8,go_game_v10,4,"duplicate_quadrant (['BL', 'TL', 'TR']), not_i..."
2,train,9,go_game_v10,4,"duplicate_quadrant (['BR', 'TL', 'TR']), not_i..."
3,train,15,go_game_v10,4,"duplicate_quadrant (['BL', 'TL', 'TR']), not_i..."
4,train,21,go_game_v10,3,wrong_count (3 vs 4)
...,...,...,...,...,...
128,test,229,go_game_v10,4,"duplicate_quadrant (['BR', 'TL', 'TR']), not_i..."
129,test,232,go_game_v10,4,"duplicate_quadrant (['BL', 'TL', 'TR']), not_i..."
130,test,7,go_game_v10,4,"duplicate_quadrant (['TL', 'TR']), not_in_4_qu..."
131,test,19,go_chess,4,"duplicate_quadrant (['BL', 'TL', 'TR']), not_i..."


## 2. Prepare Annotation Workspace

Exports flagged images and their current annotations to `data/annotate/` so the
server can serve them. Generates `images.json` → required by the annotator.

You can control which images to include via the `INCLUDE_ALL` flag below:
- `False` (default) → only flagged images
- `True` → all images with board_corner annotations (full re-annotation pass)

In [4]:
INCLUDE_ALL = False  # set True to re-annotate everything

images_out_dir = ANNOTATE_DIR / "images"
images_out_dir.mkdir(parents=True, exist_ok=True)

flagged_ids = set(zip(issues_df["split"], issues_df["image_id"].astype(str)))

entries  = []
ann_dict = {}

for split_name, ds in dataset.items():
    for sample in ds:
        iid = str(sample["image_id"])
        is_flagged = (split_name, iid) in flagged_ids

        has_corners = any(c == CATEGORIES["board_corner"] for c in sample["objects"]["category"])
        if not INCLUDE_ALL and not is_flagged:
            continue

        # Save image to disk
        fname = f"{split_name}_{iid}.jpg"
        img_path = images_out_dir / fname
        if not img_path.exists():
            sample["image"].save(img_path, format="JPEG", quality=95)

        # Build entry
        row = issues_df[(issues_df["split"] == split_name) & (issues_df["image_id"].astype(str) == iid)]
        flag_reason = str(row["issues"].values[0]) if len(row) > 0 else ""

        entries.append({
            "id": int(iid),
            "filename": fname,
            "width": sample["width"],
            "height": sample["height"],
            "source": sample.get("source_dataset", ""),
            "split": split_name,
            "flagged": is_flagged,
            "flag_reason": flag_reason,
        })

        # Original annotations
        boxes = []
        for ann_id, bbox, cat in zip(
            sample["objects"]["id"],
            sample["objects"]["bbox"],
            sample["objects"]["category"],
        ):
            x, y, w, h = bbox
            boxes.append({"id": int(ann_id), "x": float(x), "y": float(y),
                          "w": float(w), "h": float(h), "category": int(cat)})
        ann_dict[iid] = {"boxes": boxes}

# Write images.json
images_json = ANNOTATE_DIR / "images.json"
with open(images_json, "w") as f:
    json.dump({"images": entries, "annotations": ann_dict}, f, indent=2)

print(f"Exported {len(entries)} images to {images_out_dir}")
print(f"images.json written to {images_json}")

Exported 214 images to data/annotate/images
images.json written to data/annotate/images.json


## 3. Launch Annotator

Starts a Gradio-based annotation UI for correcting board_corner bounding boxes.

**Usage:**
- **Click** on the image → place a `board_corner` (up to 4)
- **Click near** an existing corner → remove it
- **Click** when 4 corners exist → move the nearest one
- Use the **dropdown** or **← Prev / Next →** buttons to navigate
- All changes are **auto-saved** to `data/annotate/corrected.json`

> ⚠ Run the cell below and open the printed URL. Press the interrupt button (■) to stop.

In [9]:
from moku.annotator import AnnotatorState, build_app

anno_state = AnnotatorState(dataset, flagged_only=False)
print(f"{anno_state.n_images} images indexed")

app = build_app(anno_state)
app.launch(server_port=7997, prevent_thread_lock=False, height=1000)

/Users/hadim/Code/libs/moku/src/moku/annotator.py:88: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  """Persist corrections to disk."""


485 images indexed
* Running on local URL:  http://127.0.0.1:7997
* To create a public link, set `share=True` in `launch()`.


## 4. Review & Apply Corrections

Load the corrections saved by the annotator and inspect what changed.
Then apply them to the dataset to produce a corrected DatasetDict.

In [ ]:
if not CORRECTED_FILE.exists():
    print(f"No corrections file found at {CORRECTED_FILE}. Run the annotator first.")
else:
    with open(CORRECTED_FILE) as f:
        corrections = json.load(f)

    print(f"Corrections for {len(corrections)} image(s):")

    rows = []
    for key, corr in corrections.items():
        split = corr.get("split", "?")
        row_idx = corr.get("row_idx", -1)
        image_id = corr.get("image_id", "?")
        n_after = sum(1 for b in corr.get("boxes", []) if b["category"] == CATEGORIES["board_corner"])
        rows.append({"key": key, "split": split, "image_id": image_id, "corners": n_after})
    display(pd.DataFrame(rows))

In [ ]:
# Apply corrections to the dataset
if CORRECTED_FILE.exists():
    corrected_dataset = apply_corner_corrections(dataset, corrections)
    print("Corrections applied. Re-run audit to verify:")
    issues_after = audit_corners(corrected_dataset)
    print(f"  Flagged images after correction: {len(issues_after)}")
    if len(issues_after) > 0:
        display(issues_after)
    else:
        print("  ✓ All images pass the audit.")

## Next Steps

After completing re-annotation:
1. In `01_Build_Dataset.ipynb`, use `build_dataset_v2()` which incorporates the
   corrections via `apply_corner_corrections()` and merges with synthetic data.
2. Push the corrected dataset as `kaya-go/moku-v2`.
3. Retrain the model with the new dataset and augmentation pipeline.